<a href="https://colab.research.google.com/github/vzavala/StatsBook/blob/master/Python/ch3_gibbs_conditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Setup.** Run this cell first: it locates the data files this notebook reads. On Google
Colab it downloads them (the `Data/` and `Support/` folders) from the book's GitHub
repository; on your computer, run the notebook from the folder that contains `Data/`.

In [1]:
import os
import sys
path = ''
if 'google.colab' in sys.modules and not os.path.isdir('Data'):
    if not os.path.isdir('statsbook'):
        os.system('git clone -q --depth 1 --filter=blob:none --sparse '
                  'https://github.com/vzavala/StatsBook.git statsbook')
        os.system('git -C statsbook sparse-checkout set Python/Data Python/Support')
    path = 'statsbook/Python/'

Empirical event probabilities for the Gibbs reactor (Victor Z, UW-Madison, 2024).

| Slides | Python | Meaning |
|---|---|---|
| $x_{\omega,1}$, $x_{\omega,2}$ | `x1`, `x2` | pressure [bar], conversion [-] |
| $\mathcal{A}_1$, $\mathcal{A}_2$ | `in_A1`, `in_A2` | indicators $\mathbf{1}[x_\omega\in\mathcal{A}_i]$ |
| $\hat{\mathbb{P}}(X\in\mathcal{A}_1\cap\mathcal{A}_2)$ | `P_A1_and_A2` | empirical joint probability |
| $\hat{\mathbb{P}}(X\in\mathcal{A}_1\mid X\in\mathcal{A}_2)$ | `P_A1_given_A2` | empirical conditional probability |

Events: $\mathcal{A}_1=\{x_1\leq 170\ \&\ x_2\geq 0.5\}$ and $\mathcal{A}_2=\{x_1\geq 120\ \&\ x_2\geq 0.5\}$.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt(path+'Data/gibbs_hightemp_class.dat')
S = len(data)
np.random.seed(1)
x1 = data[:, 1] + np.random.randn(S)          # pressure
x2 = data[:, 2] + np.random.randn(S) * 0.05   # conversion

in_A1 = (x1 <= 170) & (x2 >= 0.5)
in_A2 = (x1 >= 120) & (x2 >= 0.5)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, mask, title in ((axs[0], in_A1, r'$\mathcal{A}_1$'), (axs[1], in_A2, r'$\mathcal{A}_2$'),
                        (axs[2], in_A1 & in_A2, r'$\mathcal{A}_1\cap\mathcal{A}_2$')):
    ax.scatter(x1[~mask], x2[~mask], s=8, c='white', edgecolors='black')
    ax.scatter(x1[mask], x2[mask], s=8, c='black')
    ax.set_title(title)
    ax.set_xlabel('$x_1$ [bar]')
    ax.set_ylabel('$x_2$ [-]')
    ax.axis([50, 250, 0, 1])
    ax.grid(True)
plt.tight_layout()
plt.show()

/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40492/2659828803.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


$\hat{\mathbb{P}}(X\in\mathcal{A})=\frac{1}{S}\sum_\omega\mathbf{1}[x_\omega\in\mathcal{A}]$ and
$\hat{\mathbb{P}}(X\in\mathcal{A}_1|X\in\mathcal{A}_2)=\hat{\mathbb{P}}(X\in\mathcal{A}_1\cap\mathcal{A}_2)/\hat{\mathbb{P}}(X\in\mathcal{A}_2)$.

In [3]:
P_A1 = np.mean(in_A1)
P_A2 = np.mean(in_A2)
P_A1_and_A2 = np.mean(in_A1 & in_A2)
P_A1_or_A2 = np.mean(in_A1 | in_A2)
P_A1_given_A2 = P_A1_and_A2 / P_A2
P_A2_given_A1 = P_A1_and_A2 / P_A1
print(f"P_hat(A1) = {P_A1:.3f},  P_hat(A2) = {P_A2:.3f}")
print(f"P_hat(A1 & A2) = {P_A1_and_A2:.3f},  P_hat(A1 or A2) = {P_A1_or_A2:.3f}")
print(f"P_hat(A1 | A2) = {P_A1_given_A2:.3f},  P_hat(A2 | A1) = {P_A2_given_A1:.3f}")
print(f"Bayes' rule check: P(A2|A1) P(A1) / P(A2) = {P_A2_given_A1 * P_A1 / P_A2:.3f}")

P_hat(A1) = 0.252,  P_hat(A2) = 0.420
P_hat(A1 & A2) = 0.252,  P_hat(A1 or A2) = 0.420
P_hat(A1 | A2) = 0.600,  P_hat(A2 | A1) = 1.000
Bayes' rule check: P(A2|A1) P(A1) / P(A2) = 0.600
